In [2]:
from pathlib import Path
import qlib
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from qlib.data import D
from pipeline.utils import prints, init_log_file, get_last_trading_day
from pipeline.portfolio_builder import build_long_short_portfolio
from pipeline.display_utils import print_safe_trades
from pipeline.features import compute_all_features, momentum_label
from pipeline.daily_logger import run_daily_logging
from pipeline.performance.daily_summary import run_daily_summary
from pipeline.performance.reason_attribution import run_reason_attribution
from pipeline.execution.minimal import simulate_execution_minimal_v2
# Stability modules
from stability import run_feature_drift_monitor, run_rolling_ic_monitor, run_recent_ic_monitor


In [14]:
qlib.init(provider_uri="C:/Users/harve/.qlib/qlib_data/us_data", region="us")

D.features(["PLTR"], ["$beat_streak"], "2023-01-01", "2023-12-31")

[37020:MainThread](2026-02-23 20:17:20,455) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[37020:MainThread](2026-02-23 20:17:20,459) INFO - qlib.Initialization - [__init__.py:79] - qlib successfully initialized based on client settings.
[37020:MainThread](2026-02-23 20:17:20,462) INFO - qlib.Initialization - [__init__.py:81] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/harve/.qlib/qlib_data/us_data')}


$beat_streak
instrument datetime                
PLTR       2023-01-03      0.000000
           2023-01-04      0.000000
           2023-01-05      0.000000
           2023-01-06      0.000000
           2023-01-09      0.000000
...                             ...
           2023-12-22      0.210526
           2023-12-26      0.210526
           2023-12-27      0.210526
           2023-12-28      0.210526
           2023-12-29      0.210526

[250 rows x 1 columns]

In [3]:
# ============================================================
# CONFIG
# ============================================================
START_DATE = "2018-01-01"
END_DATE = (datetime.today() - timedelta(days=0)).strftime("%Y-%m-%d")

MODEL_PATH = "trained_model_2.pkl"

SAFE_FEATURES = [
    "$open", "$high", "$low", "$close",
    "$volume",
    "$vol_5d", "$vol_10d", "$vol_20d",
    "$rank_vol_5d", "$rank_vol_10d", "$rank_vol_20d",
    "$days_since_ipo",
]

SAFE_DF_PATH = Path("artifacts/safe_entries.parquet")
TOP_K_LONG = 20
TOP_K_SHORT = 20
IC_WINDOW_DAYS = 60

init_log_file("logs/top_long_short.log")


In [4]:
# -----------------------------
# Init Qlib
# -----------------------------
qlib.init(provider_uri="C:/Users/harve/.qlib/qlib_data/us_data", region="us")

# Load instruments
instrument_path = r"C:/Users/harve/.qlib/qlib_data/us_data/instruments/all.txt"
with open(instrument_path, "r") as f:
    instruments = [line.strip().split("\t")[0] for line in f if line.strip()]

# -----------------------------
# Load model + training columns
# -----------------------------
with open(MODEL_PATH, "rb") as f:
    saved = pickle.load(f)

model = saved["model"]
model_cols = saved["columns"]

prints(f"Loaded model from {MODEL_PATH}")
prints(f"Model expects {len(model_cols)} features")


[9784:MainThread](2026-01-17 16:02:43,666) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[9784:MainThread](2026-01-17 16:02:44,351) INFO - qlib.Initialization - [__init__.py:79] - qlib successfully initialized based on client settings.
[9784:MainThread](2026-01-17 16:02:44,352) INFO - qlib.Initialization - [__init__.py:81] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/harve/.qlib/qlib_data/us_data')}


Loaded model from trained_model_2.pkl
Model expects 12 features


In [5]:
# -----------------------------
# Load features
# -----------------------------
features = D.features(
    instruments=instruments,
    fields=SAFE_FEATURES,
    start_time=START_DATE,
    end_time=END_DATE,
)

# Feature engineering
X = features.copy()
X["$volume_log"] = np.log1p(X["$volume"])
X.drop(columns=["$volume"], inplace=True)

# Clean
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# Align columns
X = X.reindex(columns=model_cols)


In [6]:
# -----------------------------
# Predict scores
# -----------------------------
scores = model.predict(X)
df = X.copy()
df["score"] = scores

df = df.sort_index()

# === SIGNAL MOMENTUM CALCULATIONS & PRICE CRASH INDICATOR ===
df = compute_all_features(df)
# -----------------------------
# Determine latest date
# -----------------------------
dt_idx = df.index.get_level_values("datetime")
latest_date = dt_idx.max()
prints(f"Latest available date: {latest_date}")

df_today = df.loc[dt_idx == latest_date].copy()
df_today = df_today.reset_index()  # bring instrument + datetime into columns

df_today_raw = df_today.copy()
# Rename df_today columns
df_today = df_today.rename(columns={
    "$close": "close",
    "$open": "open",
    "$high": "high",
    "$low": "low",
    "$vol_20d": "vol_20d",
})

Latest available date: 2026-01-16 00:00:00


In [7]:
# ============================================================
# SAVE TODAY'S PREDICTIONS FOR ROLLING IC
# ============================================================
df_pred_today = pd.DataFrame({
    "date": pd.to_datetime(latest_date),
    "symbol": df_today["instrument"],
    "pred": df_today["score"],
})

# Save daily predictions
pred_dir = Path("stability_outputs/daily_predictions")
pred_dir.mkdir(parents=True, exist_ok=True)
df_pred_today.to_csv(pred_dir / f"preds_{latest_date.date()}.csv", index=False)

# -----------------------------
# Build long/short portfolio
# -----------------------------
portfolio = build_long_short_portfolio(
    df_today,
    top_k_long=TOP_K_LONG,
    top_k_short=TOP_K_SHORT,
    momentum_label_fn=momentum_label,
)

In [8]:
prints("\n===== LONG/SHORT PORTFOLIO =====")
for _, row in portfolio.iterrows():
    side = "LONG" if row["weight"] > 0 else "SHORT"
    level = "info"
    pad = ' '
    if row['score'] < 0:
        pad = ''
    if row["entry_human"] == "BLOCKED" or row['entry_raw'].startswith('RISKY'):
        level = "error"
    elif (
        (row["entry_human"].startswith("WATCH") and row["entry_raw"] == "SAFE")
        or row["entry_human"] == "SAFE_HI_SPREAD"
    ):
        level = "warning"
    prints(
        f"{side:<6} {row['instrument']:5s} "
        f"score={row['score']:.5f}{pad}  "
        f"mom={row['mom_label']:4s}  "
        f"crash={row['crash_label']:3s} "
        f"{row['entry_human']:14s} "  # human-safe label
        f"{row['entry_raw']:12s} "          # system view (optional)
        f"why={row['entry_reason']:20s} "
        f"weight={row['weight']:.4f}", level
    )



===== LONG/SHORT PORTFOLIO =====
LONG   ALAB  score=0.12957   mom=1x    crash=0x  BLOCKED        SAFE         why=overextended_up      weight=0.0234
LONG   COIN  score=0.08224   mom=1x    crash=0x  BLOCKED        SAFE         why=trend_misaligned     weight=0.0356
LONG   APP   score=0.08133   mom=1x    crash=4x  BLOCKED        BLOCKED      why=long_extreme_crash   weight=0.0337
LONG   RDDT  score=0.04888   mom=1x    crash=3x  BLOCKED        BLOCKED      why=long_extreme_crash   weight=0.0489
LONG   INTC  score=0.03964   mom=1x    crash=0x  SAFE_HI_SPREAD SAFE         why=high_volatility_use_spread_limit weight=0.0346
LONG   DDOG  score=0.03715   mom=1x    crash=2x  BLOCKED        RISKY        why=trend_misaligned     weight=0.0490
LONG   HOOD  score=0.03305   mom=1x    crash=3x  BLOCKED        BLOCKED      why=long_extreme_crash   weight=0.0512
LONG   AMD   score=0.03133   mom=3x    crash=0x  SAFE_HI_SPREAD SAFE         why=high_volatility_use_spread_limit weight=0.0347
LONG   MSTR  s

In [9]:
# ============================================================
# Generate safe trades
# ============================================================
safe_trades = print_safe_trades(portfolio, SAFE_DF_PATH)


=== SAFE TRADES ===

=== SAFE TRADES (sorted by LONG/SHORT → score → momentum) ===
LONG  INTC    score=0.0396  mom=1x  crash=0x  reason=high_volatility_use_spread_limit
LONG  AMD     score=0.0313  mom=3x  crash=0x  reason=high_volatility_use_spread_limit
LONG  MSTR    score=0.0300  mom=1x  crash=0x  reason=high_volatility_use_spread_limit
LONG  BABA    score=0.0294  mom=1x  crash=0x  reason=high_volatility_use_spread_limit
LONG  AVGO    score=0.0282  mom=1x  crash=0x  reason=long_clean_trend
LONG  PANW    score=0.0191  mom=1x  crash=0x  reason=long_clean_trend
LONG  C       score=0.0190  mom=1x  crash=0x  reason=long_clean_trend
SHORT AAPL    score=-0.0131  mom=1x  crash=0x  reason=short_clean_trend
SHORT SHOP    score=-0.0185  mom=1x  crash=2x  reason=short_clean_trend


In [10]:
# ============================================================
# FEATURE DRIFT MONITOR
# ============================================================
train_sample = pd.read_parquet("artifacts/train_features_sample.parquet")
run_feature_drift_monitor(
    train_feature_sample=train_sample,
    daily_features=df_today_raw.set_index("instrument")[model_cols],
    out_dir="stability_outputs/feature_drift",
    date_str=str(latest_date.date()),
)


[DRIFT] Running drift monitor for 2026-01-16
[DRIFT] Drift CSV saved to: stability_outputs\feature_drift\feature_drift_2026-01-16.csv
[DRIFT] Alerts CSV saved to: stability_outputs\feature_drift\feature_drift_alerts_2026-01-16.csv
[DRIFT] Summary: {'date': '2026-01-16', 'n_features': 12, 'n_alerts': 10, 'max_psi': 1.4715067372765251, 'max_ks': 0.7659574468085106}


{'date': '2026-01-16',
 'n_features': 12,
 'n_alerts': 10,
 'max_psi': 1.4715067372765251,
 'max_ks': 0.7659574468085106}

In [ ]:
# -----------------------------
# IC evaluation over recent window
# -----------------------------
run_recent_ic_monitor(
    df=df,
    dt_idx=dt_idx,
    instruments=instruments,
    start_date=START_DATE,
    end_date=END_DATE,
    window=IC_WINDOW_DAYS,
)


In [ ]:
# ============================================================
# ROLLING IC STABILITY (requires labels to exist)
# ============================================================
# Rolling IC monitor
run_rolling_ic_monitor(
    instruments=instruments,
    start_date=START_DATE,
    end_date=END_DATE,
    pred_dir="stability_outputs/daily_predictions",
    window=20,
)



[IC] Running rolling IC monitor...
[IC] Rolling IC summary: {'last_date': '2026-01-09', 'IC_last': -0.13361964184425898, 'IC_20_last': None, 'IC_vol_20_last': None, 'n_alerts_total': 0}


{'last_date': '2026-01-09',
 'IC_last': -0.13361964184425898,
 'IC_20_last': None,
 'IC_vol_20_last': None,
 'n_alerts_total': 0}

In [ ]:
# ============================================================
# DAILY LOGGING AND PERFORMANCE SUMMARY
# ============================================================
today = get_last_trading_day()
run_daily_logging(today)
run_daily_summary()
run_reason_attribution()


=== DAILY LOGGING (2026-01-16) ===
Loaded SAFE entries: 9 symbols
Loading today's prices…
Logging entries (duplicate-safe)…


c:\ws\qlib\pipeline\entry_logger.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated = pd.concat([log, df], ignore_index=True)


Entry log now contains 56 total rows
Loading full price history for forward returns…
Updating forward returns…
Done.
Forward returns updated for 2026-01-16.


      DAILY SUMMARY REPORT

=== SAFE TRADES SUMMARY ===
Total SAFE trades: 9
Long: 7   Short: 2

--- Reason Codes ---
high_volatility_use_spread_limit: 4
long_clean_trend: 3
short_clean_trend: 2

--- Momentum Buckets ---
-0.008873720152889286: 1
0.0257985802236036: 1
0.002782724554226231: 1
-0.002668540786576256: 1
-0.003956437148442956: 1
0.0003237582155966208: 1
0.0058954020752847: 1
0.0038206523985020724: 1
-0.0005670727110343166: 1

--- Crash Flags ---
0: 8
2: 1

=== DRIFT SUMMARY ===
No drift summary available.

=== IC SUMMARY ===
No IC summary available.

=== END OF DAILY SUMMARY ===


===== REASON-CODE ATTRIBUTION =====

Reason: long_clean_trend
Total trades: 27
  5d: n_valid=5, hit=0.400, avg=-0.0107, med=-0.0104, std=0.0333
  10d: n_valid=0 (no matured trades yet)
  20d: n_valid=0 (no matured trades yet)
  60d: n_valid=0

In [ ]:
# ============================================================
# Simulate execution of safe trades
# ============================================================
# === ATTACH WEIGHTS FIRST (this fixes the KeyError) ===
safe_trades = safe_trades.merge(
    portfolio[["instrument", "weight"]],
    left_on="symbol",
    right_on="instrument",
    how="left"
).drop(columns=["instrument"])

safe_trades = safe_trades.rename(columns={"direction": "side"})

    # Simulate execution
PORTFOLIO_NOTIONAL = 100000
# === SYNTHETIC MICROSTRUCTURE FIELDS ===
# price = today's close
safe_trades["price"] = df_today.set_index("instrument").loc[safe_trades["symbol"], "close"].values

# spread model: 2 bps + volatility adjustment
safe_trades["spread_bps"] = 2 + 0.1 * safe_trades["crash"]  # or use vol_20d if you prefer

# volatility proxy
safe_trades["vol_daily"] = df_today.set_index("instrument").loc[safe_trades["symbol"], "vol_20d"].values

# ADV proxy: use today's volume_log exponentiated (approx)
# since $volume was dropped, we approximate ADV from vol_20d
safe_trades["adv_shares"] = 1e6 * (1 + safe_trades["vol_daily"])  # simple synthetic ADV

safe_exec = simulate_execution_minimal_v2(
    safe_trades,
    portfolio_notional=PORTFOLIO_NOTIONAL
)
safe_exec.to_parquet(f"execution_outputs/safe_exec_{latest_date.date()}.parquet", index=False)


In [ ]:
cols_to_keep = [
    "symbol", "side", "weight", "score", "reason",
    "price", "effective_price", "slippage_bps",
    "order_size_shares", "fill_fraction", "executed_notional"
]

safe_exec_log = safe_exec[cols_to_keep].copy()
safe_exec_log["date"] = latest_date

prints("=== EXECUTION v2 SUMMARY ===")
prints(f"Avg slippage (bps): {safe_exec['slippage_bps'].mean():.2f}")
prints(f"Avg fill fraction: {safe_exec['fill_fraction'].mean():.2f}")
prints(f"Total executed notional: {safe_exec['executed_notional'].sum():,.0f}")

=== EXECUTION v2 SUMMARY ===
Avg slippage (bps): 2.22
Avg fill fraction: 0.95
Total executed notional: 41,281


,symbol,side,score,momentum,crash,reason,weight,price,spread_bps,vol_daily,adv_shares,order_notional,order_size_shares,rel_adv,slippage_bps,fill_fraction,effective_price,executed_notional
0,INTC,LONG,0.039637,-0.008874,0,high_volatility_use_spread_limit,0.034598,1.204659,2.0,0.033609,1.033609e+06,3459.827881,2872.039062,0.002779,2.306977,0.952445,1.204937,3295.294627
1,AMD,LONG,0.031325,0.025799,0,high_volatility_use_spread_limit,0.034720,21.105648,2.0,0.033491,1.033491e+06,3472.041016,164.507675,0.000159,2.175412,0.953254,21.110239,3309.737481
2,MSTR,LONG,0.030049,0.002783,0,high_volatility_use_spread_limit,0.025992,13.035516,2.0,0.044738,1.044738e+06,2599.171875,199.391571,0.000191,2.233231,0.950995,13.038427,2471.799993
3,BABA,LONG,0.029444,-0.002669,0,high_volatility_use_spread_limit,0.037874,0.947701,2.0,0.030702,1.030702e+06,3787.368652,3996.373779,0.003877,2.347378,0.952696,0.947924,3608.212369
4,AVGO,LONG,0.028210,-0.003956,0,long_clean_trend,0.075643,16.329969,2.0,0.015372,1.015372e+06,7564.271973,463.214081,0.000456,2.099672,0.956789,16.333398,7237.409781


In [ ]:
# === APPEND EXECUTION-AWARE ENTRIES TO SAFE_DF_PATH ===
SAFE_DF_PATH = Path("artifacts/safe_entries.parquet")

# Load existing SAFE entries log
if SAFE_DF_PATH.exists():
    log = pd.read_parquet(SAFE_DF_PATH)
else:
    log = pd.DataFrame()

print(log.head())
# Drop duplicates: (date, symbol)
existing_pairs = set(zip(log["date"], log["symbol"]))
new_rows = [
    i for i, row in safe_exec_log.iterrows()
    if (row["date"], row["symbol"]) not in existing_pairs
]

df_new = safe_exec_log.loc[new_rows]

# Append
updated = pd.concat([log, df_new], ignore_index=True)

# Save
updated.to_parquet(SAFE_DF_PATH, index=False)

prints(f"Logged {len(df_new)} execution-aware entries.")
prints(f"SAFE entries log now contains {len(updated)} rows.")

  symbol direction     score  momentum  crash  \
0   INTC      LONG  0.039637 -0.008874      0   
1    AMD      LONG  0.031325  0.025799      0   
2   MSTR      LONG  0.030049  0.002783      0   
3   BABA      LONG  0.029444 -0.002669      0   
4   AVGO      LONG  0.028210 -0.003956      0   

                             reason  
0  high_volatility_use_spread_limit  
1  high_volatility_use_spread_limit  
2  high_volatility_use_spread_limit  
3  high_volatility_use_spread_limit  
4                  long_clean_trend  


KeyError: 'date'